# `CVXPY` Examples

This notebook contains two fully worked constrained-optimization examples. The diet problem introduces the basic `CVXPY` workflow, and the Crossy Road kiosk problem shows how a state variable connects decisions across timesteps. Run the cells from top to bottom.

In [1]:
import sys
import cvxpy as cvx
import numpy as np
import pandas as pd
from IPython.display import display

if sys.version_info[:3] != (3, 10, 6):
    raise RuntimeError("Select your Python 3.10.6 CS410 environment.")

## Example 1: Minimum-Cost Diet

Choose servings of food that meet minimum nutrient requirements at the lowest cost.

Let $x_i$ be servings of food $i$, $c_i$ its cost, $A_{ji}$ the amount of nutrient $j$ in that food, and $b_j$ the minimum requirement for nutrient $j$. The model is

$$\min_x c^T x \qquad \text{subject to } Ax\ge b,\quad x\ge 0.$$

We first build the model with individual variables and loops, then show an equivalent vectorized formulation.

In [2]:
foods = ["apple", "banana", "carrot", "chicken", "rice", "milk"]
nutrients = ["calories", "protein", "carbs", "fat"]

food_nutrients = {
    "apple":   {"calories": 95,  "protein": 0.3, "carbs": 25, "fat": 0.4},
    "banana":  {"calories": 105, "protein": 1.3, "carbs": 27, "fat": 0.4},
    "carrot":  {"calories": 41,  "protein": 0.9, "carbs": 9,  "fat": 0.2},
    "chicken": {"calories": 142, "protein": 31,  "carbs": 0,  "fat": 3.6},
    "rice":    {"calories": 130, "protein": 2.6, "carbs": 28, "fat": 0.3},
    "milk":    {"calories": 100, "protein": 8,   "carbs": 12, "fat": 3.2},
}
minimum_nutrients = {"calories": 2000, "protein": 50, "carbs": 300, "fat": 65}
food_costs = {"apple": 0.5, "banana": 0.3, "carrot": 0.2, "chicken": 2.5, "rice": 0.8, "milk": 1.2}

### List-and-Loop Form

This form mirrors the verbal problem closely. Each food gets its own scalar variable, and each nutrient gets its own constraint.

In [3]:
# Decision variables
serving_variables = [cvx.Variable(nonneg=True) for food in foods]

# Objective
total_cost = sum(
    food_costs[food] * serving_variables[i]
    for i, food in enumerate(foods)
)
objective = cvx.Minimize(total_cost)

# Constraints
constraints = []
for nutrient in nutrients:
    nutrient_total = sum(
        food_nutrients[food][nutrient] * serving_variables[i]
        for i, food in enumerate(foods)
    )
    constraints.append(nutrient_total >= minimum_nutrients[nutrient])

diet_list_problem = cvx.Problem(objective, constraints)
diet_list_problem.solve(solver="CLARABEL")

list_solution = np.array([variable.value for variable in serving_variables])
print("Status:", diet_list_problem.status)
print("Minimum cost:", diet_list_problem.value)
display(pd.Series(list_solution, index=foods, name="servings").to_frame())

Status: optimal
Minimum cost: 24.70588235512291


,servings
apple,2.070504e-09
banana,2.205882e+00
carrot,5.526107e-09
chicken,2.984753e-10
rice,5.981085e-10
milk,2.003676e+01


### Vectorized Form

The same model can be written more compactly using a decision vector and matrix multiplication. The ordering of rows and columns must remain consistent.

In [ ]:
cost_vector = np.array([food_costs[food] for food in foods])
nutrient_matrix = np.array([
    [food_nutrients[food][nutrient] for food in foods]
    for nutrient in nutrients
])
minimum_vector = np.array([minimum_nutrients[nutrient] for nutrient in nutrients])

servings = cvx.Variable(len(foods), nonneg=True)
objective = cvx.Minimize(cost_vector @ servings)
constraints = [nutrient_matrix @ servings >= minimum_vector]
diet_vector_problem = cvx.Problem(objective, constraints)
diet_vector_problem.solve(solver="CLARABEL")

print("Status:", diet_vector_problem.status)
print("Minimum cost:", diet_vector_problem.value)
display(pd.Series(servings.value, index=foods, name="servings").to_frame())

Status: optimal
Minimum cost: 24.70588235512291


,servings
apple,2.070504e-09
banana,2.205882e+00
carrot,5.526107e-09
chicken,2.984753e-10
rice,5.981085e-10
milk,2.003676e+01


## Example 2: Crossy Road Keychain Kiosk

This example introduces a **state variable**, whose value at one timestep carries information into the next timestep.

A campus kiosk sells Chicken and Lion keychains. Demand for the next three days is known, but supplier prices change each day. The kiosk begins with no inventory and can hold at most 10 keychains at the end of each day. We must meet all demand while minimizing purchasing cost.

### Data

| Day | Demand | Chicken price | Lion price |
|:---:|---|:---:|:---:|
| 1 | 2 Chickens, 3 Lions | $1 | $2 |
| 2 | 8 Chickens, 6 Lions | $5 | $10 |
| 3 | 4 Chickens, 2 Lions | $2 | $3 |

Let $D_{i,t}$ denote demand for item $i$ on day $t$, and let $P_{i,t}$ denote its purchase price.

In [ ]:
items = ["Chicken", "Lion"]
days = ["Day 1", "Day 2", "Day 3"]

# Rows correspond to items; columns correspond to days.
demand = np.array([
    [2, 8, 4], # Chicken
    [3, 6, 2], # Lion
], dtype=float)
prices = np.array([
    [1, 5, 2],  # Chicken
    [2, 10, 3], # Lion
], dtype=float)

print("Demand for each item on each day:")
display(pd.DataFrame(demand, index=items, columns=days).rename_axis("Demand"))
print("Price for each item on each day:")
display(pd.DataFrame(prices, index=items, columns=days).rename_axis("Price"))

,Day 1,Day 2,Day 3
Demand,,,
Chicken,2.0,8.0,4.0
Lion,3.0,6.0,2.0


,Day 1,Day 2,Day 3
Price,,,
Chicken,1.0,5.0,2.0
Lion,2.0,10.0,3.0


### Mathematical Formulation

For each item $i$ and day $t$, define:

- $B_{i,t}\ge 0$: number of keychains bought on day $t$.
- $I_{i,t}\ge 0$: inventory remaining at the **end** of day $t$.

The inventory balance connects consecutive days:

$$I_{i,t}=I_{i,t-1}+B_{i,t}-D_{i,t}.$$

In words:

$$\text{inventory left tonight}=\text{inventory from yesterday}+\text{bought today}-\text{sold today}.$$

The complete model minimizes

$$\sum_{t=1}^{3}\sum_i P_{i,t}B_{i,t},$$

subject to the inventory balance above, the daily capacity constraint

$$\sum_i I_{i,t}\le 10,$$

the initial condition $I_{i,0}=0$, and nonnegativity of every purchase and inventory variable.

In [6]:
item_count, day_count = demand.shape

# Decision variables
bought = cvx.Variable((item_count, day_count), nonneg=True)
# inventory[i, 0] is initial inventory for item i.
# inventory[i, t + 1] is inventory after day t (Python days start at 0).
inventory = cvx.Variable((item_count, day_count + 1), nonneg=True)

# This limit can be changed between solves without rebuilding the model.
kiosk_capacity = cvx.Parameter(nonneg=True)
kiosk_capacity.value = 10

# Objective
total_cost = 0
for t in range(day_count):
    for i in range(item_count):
        total_cost += prices[i, t] * bought[i, t]
objective = cvx.Minimize(total_cost)

# Constraints
constraints = []
for i in range(item_count):
    constraints.append(inventory[i, 0] == 0)

for t in range(day_count):
    total_inventory = 0
    for i in range(item_count):
        constraints.append(
            inventory[i, t + 1] == inventory[i, t] + bought[i, t] - demand[i, t]
        )
        total_inventory += inventory[i, t + 1]
    constraints.append(total_inventory <= kiosk_capacity)

kiosk_problem = cvx.Problem(objective, constraints)
kiosk_problem.solve(solver="CLARABEL")

if kiosk_problem.status not in (cvx.OPTIMAL, cvx.OPTIMAL_INACCURATE):
    raise ValueError(f"No usable kiosk plan: {kiosk_problem.status}")

In [ ]:
purchase_table = pd.DataFrame(bought.value, index=items, columns=days)
inventory_table = pd.DataFrame(inventory.value[:, 1:], index=items, columns=days)

print("Status:", kiosk_problem.status)
print(f"Minimum total cost: ${kiosk_problem.value:.2f}")
print("Keychains bought:")
display(purchase_table.round(6))
print("End-of-day inventory:")
display(inventory_table.round(6))

Status: optimal
Minimum total cost: $58.00
Keychains bought:


,Day 1,Day 2,Day 3
Chicken,6.0,4.0,4.0
Lion,9.0,0.0,2.0


End-of-day inventory:


,Day 1,Day 2,Day 3
Chicken,4.0,0.0,0.0
Lion,6.0,0.0,0.0
